In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, classification_report

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 1. Cargar Datos de Predicciones

In [2]:
# Cargar archivo de predicciones
predictions_file = '../outputs/resnet50_20260103_235009/prediction_scores_test.csv'
df = pd.read_csv(predictions_file)

print(f"Total de muestras: {len(df)}")
print(f"\nColumnas disponibles: {df.columns.tolist()}")
print(f"\nPrimeras filas:")
df.head()

Total de muestras: 17125

Columnas disponibles: ['true_label', 'predicted_score']

Primeras filas:


,true_label,predicted_score
0,0,0.883439
1,0,0.801483
2,0,0.474656
3,0,0.259963
4,0,0.501993


## 2. Calcular Métricas por Clase

In [6]:
# Crear etiquetas predichas a partir de los scores (umbral 0.5)
df['predicted_label_binary'] = (df['predicted_score'] >= 0.5).astype(int)

# Mapear 0/1 a benign/malignant
label_mapping = {0: 'benign', 1: 'malignant'}
df['true_label_name'] = df['true_label'].map(label_mapping)
df['predicted_label_name'] = df['predicted_label_binary'].map(label_mapping)

# Obtener valores verdaderos y predicciones
y_true = df['true_label_name'].values
y_pred = df['predicted_label_name'].values

# Calcular métricas por clase
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=['benign', 'malignant']
)

# Crear DataFrame con las métricas
metrics_df = pd.DataFrame({
    'Class': ['benign', 'malignant'],
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print("\n" + "="*70)
print("MÉTRICAS POR CLASE")
print("="*70)
print(metrics_df.to_string(index=False))
print("="*70)

# Mostrar reporte de clasificación completo
print("\n" + "="*70)
print("REPORTE DE CLASIFICACIÓN COMPLETO")
print("="*70)
print(classification_report(y_true, y_pred, target_names=['benign', 'malignant']))
print("="*70)


MÉTRICAS POR CLASE
    Class  Precision   Recall  F1-Score  Support
   benign   0.969961 0.908675  0.938318    15067
malignant   0.542857 0.793975  0.644830     2058

REPORTE DE CLASIFICACIÓN COMPLETO
              precision    recall  f1-score   support

      benign       0.97      0.91      0.94     15067
   malignant       0.54      0.79      0.64      2058

    accuracy                           0.89     17125
   macro avg       0.76      0.85      0.79     17125
weighted avg       0.92      0.89      0.90     17125



## 3. Visualización de Métricas por Clase

In [ ]:
# Gráfico de barras agrupadas para las métricas
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x = np.arange(len(metrics_df['Class']))
width = 0.25

bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision', color='#3498db', alpha=0.8)
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall', color='#2ecc71', alpha=0.8)
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#e74c3c', alpha=0.8)

# Añadir valores sobre las barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Clase', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Métricas por Clase - Comparación', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Class'])
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Gráfico de Radar para Comparación de Métricas

In [ ]:
from math import pi

# Preparar datos para gráfico de radar
categories = ['Precision', 'Recall', 'F1-Score']
N = len(categories)

# Crear figura
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(projection='polar'))

# Ángulos para cada métrica
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# Colores para cada clase
colors = ['#3498db', '#e74c3c']
class_names = ['benign', 'malignant']

for idx, (ax, class_name, color) in enumerate(zip(axes, class_names, colors)):
    # Obtener valores de la clase
    values = metrics_df[metrics_df['Class'] == class_name][['Precision', 'Recall', 'F1-Score']].values.flatten().tolist()
    values += values[:1]
    
    # Dibujar
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=class_name)
    ax.fill(angles, values, alpha=0.25, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
    ax.set_title(f'Clase: {class_name.upper()}\n(Support: {int(metrics_df[metrics_df["Class"] == class_name]["Support"].values[0])})',
                 fontsize=12, fontweight='bold', pad=20)
    ax.grid(True)

plt.tight_layout()
plt.show()

## 5. Heatmap de Métricas

In [ ]:
# Crear matriz para heatmap
heatmap_data = metrics_df[['Precision', 'Recall', 'F1-Score']].T
heatmap_data.columns = metrics_df['Class']

# Crear heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            cbar_kws={'label': 'Score'}, vmin=0, vmax=1,
            linewidths=2, linecolor='white', ax=ax)

ax.set_title('Heatmap de Métricas por Clase', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Clase', fontsize=12, fontweight='bold')
ax.set_ylabel('Métrica', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Distribución de Scores de Predicción por Clase

In [ ]:
# Visualizar distribución de scores para cada clase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, true_class in enumerate(['benign', 'malignant']):
    # Filtrar datos por clase verdadera
    class_data = df[df['true_label_name'] == true_class]
    
    # Scores de predicción para malignant (predicted_score representa la probabilidad de malignant)
    scores = class_data['predicted_score'].values
    
    # Histogram
    axes[idx].hist(scores, bins=50, alpha=0.7, color=colors[idx], edgecolor='black')
    axes[idx].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Umbral = 0.5')
    axes[idx].set_xlabel('Score de Malignidad', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'Distribución de Scores - Clase Verdadera: {true_class.upper()}\n(n={len(class_data)})',
                       fontsize=12, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Análisis de Soporte (Support) por Clase

In [ ]:
# Gráfico de barras para el soporte
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Barras de soporte
bars = ax1.bar(metrics_df['Class'], metrics_df['Support'], color=colors, alpha=0.7, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_xlabel('Clase', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Muestras', fontsize=12, fontweight='bold')
ax1.set_title('Soporte por Clase', fontsize=14, fontweight='bold', pad=20)
ax1.grid(axis='y', alpha=0.3)

# Gráfico de pie para proporciones
ax2.pie(metrics_df['Support'], labels=metrics_df['Class'], autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Distribución de Muestras por Clase', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# Calcular ratio de desbalance
ratio = metrics_df['Support'].max() / metrics_df['Support'].min()
print(f"\nRatio de desbalance de clases: {ratio:.2f}:1")
print(f"La clase 'benign' tiene {ratio:.2f} veces más muestras que 'malignant'")

## 8. Resumen Ejecutivo

In [ ]:
print("\n" + "="*70)
print("RESUMEN EJECUTIVO - MÉTRICAS POR CLASE")
print("="*70)

for _, row in metrics_df.iterrows():
    print(f"\nClase: {row['Class'].upper()}")
    print(f"  • Precision:  {row['Precision']:.4f} ({row['Precision']*100:.2f}%)")
    print(f"  • Recall:     {row['Recall']:.4f} ({row['Recall']*100:.2f}%)")
    print(f"  • F1-Score:   {row['F1-Score']:.4f} ({row['F1-Score']*100:.2f}%)")
    print(f"  • Support:    {int(row['Support']):,} muestras")

print("\n" + "="*70)
print("OBSERVACIONES:")
print("="*70)

# Análisis automático
benign_metrics = metrics_df[metrics_df['Class'] == 'benign'].iloc[0]
malignant_metrics = metrics_df[metrics_df['Class'] == 'malignant'].iloc[0]

print(f"\n1. Clase BENIGN:")
print(f"   - Excelente precision ({benign_metrics['Precision']:.3f}): pocas muestras benignas son clasificadas como malignas.")
print(f"   - Recall de {benign_metrics['Recall']:.3f}: detecta el {benign_metrics['Recall']*100:.1f}% de casos benignos.")

print(f"\n2. Clase MALIGNANT:")
print(f"   - Recall alto ({malignant_metrics['Recall']:.3f}): detecta el {malignant_metrics['Recall']*100:.1f}% de casos malignos (importante en medicina).")
print(f"   - Precision moderada ({malignant_metrics['Precision']:.3f}): algunos casos benignos son clasificados como malignos.")

print(f"\n3. Desbalance de clases:")
print(f"   - Ratio de {ratio:.2f}:1 (benign:malignant)")
print(f"   - El modelo favorece la clase mayoritaria (benign).")

print("\n" + "="*70)